<a href="https://colab.research.google.com/github/bsheese/cs377/blob/18_classification_redesign/18_classification/18_1_Classification_Basics/18_1_4_Advanced_Multiclass_Extensions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Multiclass & Extensions

**Author:** Brad Sheese

---

## Introduction

This notebook covers advanced multiclass topics:

- **OvR vs OvO**: Two strategies for multiclass classification
- **Decision Boundaries**: Visualizing how classifiers separate classes
- **Macro vs Weighted Metrics**: Averaging strategies for K classes
- **Classifier Comparison**: Logistic Regression, Decision Tree, Random Forest

### Learning Objectives
By the end of this notebook, you will:
1. Explain OvR and OvO multiclass strategies
2. Visualize decision boundaries in 2D
3. Interpret macro vs weighted metrics
4. Compare different classifiers for multiclass problems

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Load wine data
wine = load_wine()
X_wine = wine.data
y_wine = wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=42, stratify=y_wine
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Wine Dataset: {len(X_train)} train, {len(X_test)} test")
print(f"Classes: {np.unique(y_wine)}")
print(f"Distribution: {np.bincount(y_wine)}")

## OvR vs OvO: Two Multiclass Strategies

When extending binary classifiers to K classes:

### One-vs-Rest (OvR)
- Train K binary classifiers
- Classifier k: "Is class k?" vs "Is NOT class k?"
- Requires K models
- For K=3: C0=(C0 vs C1,C2), C1=(C1 vs C0,C2), C2=(C2 vs C0,C1)

### One-vs-One (OvO)
- Train K(K-1)/2 binary classifiers
- Classifier (i,j): "Is class i?" vs "Is class j?"
- For K=3: (C0 vs C1), (C0 vs C2), (C1 vs C2) = 3 models
- Each model only sees samples of two classes

**Trade-offs:**
- OvR: Fewer models (K), but each trained on all data
- OvO: More models (K(K-1)/2), but each trained on 2/K of data

In [ ]:
# Compare OvR vs OvO
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier

base_clf = LogisticRegression(max_iter=1000, random_state=42)

# OvR
ovr = OneVsRestClassifier(base_clf)
ovr.fit(X_train_s, y_train)
acc_ovr = accuracy_score(y_test, ovr.predict(X_test_s))
print(f"One-vs-Rest (K models): {acc_ovr:.3f}")

# OvO
ovo = OneVsOneClassifier(base_clf)
ovo.fit(X_train_s, y_train)
acc_ovo = accuracy_score(y_test, ovo.predict(X_test_s))
print(f"One-vs-One (K(K-1)/2 models): {acc_ovo:.3f}")

# Softmax (sklearn default)
softmax = LogisticRegression(max_iter=1000, random_state=42)
softmax.fit(X_train_s, y_train)
acc_softmax = accuracy_score(y_test, softmax.predict(X_test_s))
print(f"Softmax (single model): {acc_softmax:.3f}")

print("\nFor logistic regression, all three approaches give identical results!")
print("OvO/OvR differ when base classifier is not calibrated.")

## Decision Boundaries

Let's visualize how classifiers separate classes in 2D (using PCA to reduce dimensionality).

In [ ]:
# Reduce to 2D with PCA for visualization
pca = PCA(n_components=2)
X_train_2d = pca.fit_transform(X_train_s)
X_test_2d = pca.transform(X_test_s)

print(f"PCA explained variance: {pca.explained_variance_ratio_.sum()*100:.1f}%")

# Train classifiers on 2D data
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=3, random_state=42),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5)
}

for name, clf in classifiers.items():
    clf.fit(X_train_2d, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test_2d))
    print(f"{name} (2D): {acc:.3f}")

In [ ]:
# Plot decision boundaries
def plot_decision_boundary(clf, X, y, ax, title):
    h = 0.02  # step size
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlBu)
    ax.scatter(X[y==0, 0], X[y==0, 1], c='red', label='Class 0', edgecolors='k')
    ax.scatter(X[y==1, 0], X[y==1, 1], c='blue', label='Class 1', edgecolors='k')
    ax.scatter(X[y==2, 0], X[y==2, 1], c='green', label='Class 2', edgecolors='k')
    ax.set_title(title)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for (name, clf), ax in zip(classifiers.items(), axes):
    plot_decision_boundary(clf, X_train_2d, y_train, ax, name)
    acc = accuracy_score(y_train, clf.predict(X_train_2d))
    ax.text(0.05, 0.95, f'Train Acc: {acc:.2f}', transform=ax.transAxes, fontsize=9)

plt.tight_layout()
plt.show()

## Confusion Matrix for Multiclass

With K classes, confusion matrix is KxK. Perfect model: diagonal only.

In [ ]:
# Train best model on full features
best_model = LogisticRegression(max_iter=1000, random_state=42)
best_model.fit(X_train_s, y_train)
y_pred = best_model.predict(X_test_s)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1, 2])
ax.set_yticks([0, 1, 2])
ax.set_xticklabels(['Class 0', 'Class 1', 'Class 2'])
ax.set_yticklabels(['Class 0', 'Class 1', 'Class 2'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix: Wine Classification')

for i in range(3):
    for j in range(3):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=16, fontweight='bold')

plt.colorbar(im)
plt.tight_layout()
plt.show()

print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred):.3f}")

## Macro vs Weighted Metrics

With K classes, how do we average precision/recall/F1?

**Macro**: Average of per-class metrics (each class equally weighted)
- Good when you care equally about all classes

**Weighted**: Average weighted by class frequency
- Better reflects overall performance on imbalanced data

**Micro**: Global TP, FP, FN then compute metric
- Equivalent to accuracy for multiclass

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

print("Classification Report:")
print(classification_report(y_test, y_pred, digits=3))

# Compute metrics
metrics = ['Precision', 'Recall', 'F1-Score']
macro = [
    precision_score(y_test, y_pred, average='macro'),
    recall_score(y_test, y_pred, average='macro'),
    f1_score(y_test, y_pred, average='macro')
]
weighted = [
    precision_score(y_test, y_pred, average='weighted'),
    recall_score(y_test, y_pred, average='weighted'),
    f1_score(y_test, y_pred, average='weighted')
]

# Visualize
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(3)
width = 0.35
ax.bar(x - width/2, macro, width, label='Macro', alpha=0.8, color='steelblue')
ax.bar(x + width/2, weighted, width, label='Weighted', alpha=0.8, color='coral')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel('Score')
ax.set_title('Macro vs Weighted Averaging')
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

## Key Takeaways

**Multiclass Strategies:**
- OvR: K binary classifiers (one per class)
- OvO: K(K-1)/2 binary classifiers (pairwise)
- Softmax: Single model with K outputs (preferred for most cases)

**Decision Boundaries:**
- Logistic Regression: Linear boundaries
- Decision Tree: Axis-aligned rectangular regions
- KNN: Non-linear, adapts to local data density

**Metric Averaging:**
- Macro: Equal weight per class
- Weighted: Weight by class frequency
- Micro: Pool all TP/FP/FN globally